In [ ]:
# Install required dependencies
!pip install -q kaggle-benchmarks numpy

# Learning Curves BenchmarkTests how performance improves with increasing training examples.Uses procedurally generated rule systems.**Cognitive Science**: Power Law of Practice (Newell & Rosenbloom, 1981)**Key innovation**: Novel rule systems that cannot be in training data

In [ ]:
"""Novel Rule System Generator for Learning Benchmarks.Generates procedural rule systems that cannot be in training data.Each system defines a mapping from inputs to outputs via a chainof deterministic rules. Difficulty is controlled by:- Number of rules- Number of input features- Rule interaction complexity (independent vs. chained)Systems are seeded for reproducibility across runs."""import randomimport hashlibfrom dataclasses import dataclass, field@dataclassclass RuleSystem:    """A generated rule system with examples."""    name: str    description: str    rules: list[str]    examples: list[dict]  # {"input": str, "output": str}    test_items: list[dict]  # {"input": str, "output": str}    difficulty: int  # 1-3    n_rules: int    domain: str  # "symbol", "language", "number"def _make_rng(seed: str) -> random.Random:    h = int(hashlib.sha256(seed.encode()).hexdigest(), 16)    return random.Random(h)def generate_symbol_system(seed: str = "sym_default", difficulty: int = 1) -> RuleSystem:    """    Generate a symbol transformation rule system.    Input: sequence of symbols (e.g., "△ ○ □")    Rules: transformations (e.g., "△ followed by ○ becomes ★")    Output: transformed sequence    """    rng = _make_rng(seed)    shapes = ["△", "○", "□", "◇", "★", "⬡", "⬟", "▽"]    colors = ["red", "blue", "green", "yellow"]    if difficulty == 1:        # Simple 1-to-1 substitution        src = rng.sample(shapes[:4], 3)        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes[4:])]        mapping = dict(zip(src, dst[:3]))        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]        rules.append("All other symbols stay the same")        def apply_rules(seq):            return [mapping.get(s, s) for s in seq]    elif difficulty == 2:        # Context-dependent: pairs matter        src = rng.sample(shapes[:5], 4)        dst = rng.sample(shapes[4:], 3) + [rng.choice(shapes)]        mapping = dict(zip(src[:3], dst[:3]))        pair_rule = (src[0], src[1], dst[3])  # "X followed by Y becomes Z"        rules = [f"Replace {s} with {d}" for s, d in mapping.items()]        rules.append(f"EXCEPTION: {pair_rule[0]} followed by {pair_rule[1]} → both become {pair_rule[2]}")        rules.append("All other symbols stay the same")        def apply_rules(seq):            result = []            i = 0            while i < len(seq):                if i + 1 < len(seq) and seq[i] == pair_rule[0] and seq[i + 1] == pair_rule[1]:                    result.extend([pair_rule[2], pair_rule[2]])                    i += 2                else:                    result.append(mapping.get(seq[i], seq[i]))                    i += 1            return result    else:  # difficulty == 3        # Multi-pass with conditional rules        src = rng.sample(shapes[:6], 5)        dst = rng.sample(shapes, 5)        mapping1 = {src[0]: dst[0], src[1]: dst[1]}        mapping2 = {dst[0]: dst[2]}  # Chain: src[0] → dst[0] → dst[2]        cond = src[2]  # If this symbol is present, apply extra rule        extra_map = {src[3]: dst[3]}        rules = [            f"Pass 1: Replace {s} with {d}" for s, d in mapping1.items()        ]        rules.append(f"Pass 2: Replace {list(mapping2.keys())[0]} with {list(mapping2.values())[0]}")        rules.append(f"IF the sequence contains {cond}: also replace {src[3]} with {dst[3]}")        rules.append("All other symbols stay the same throughout")        def apply_rules(seq):            # Pass 1            result = [mapping1.get(s, s) for s in seq]            # Pass 2            result = [mapping2.get(s, s) for s in result]            # Conditional            if cond in seq:  # Check original sequence                result = [extra_map.get(s, s) for s in result]            return result    # Generate examples    all_items = []    for _ in range(25):        length = rng.randint(3, 6)        seq = [rng.choice(shapes[:5]) for _ in range(length)]        output = apply_rules(seq)        all_items.append({"input": " ".join(seq), "output": " ".join(output)})    # Deduplicate by input    seen = set()    unique_items = []    for item in all_items:        if item["input"] not in seen:            seen.add(item["input"])            unique_items.append(item)    rng.shuffle(unique_items)    n_examples = min(15, len(unique_items) - 5)    examples = unique_items[:n_examples]    test_items = unique_items[n_examples:n_examples + 5]    return RuleSystem(        name=f"SymbolTransform-{seed}",        description="Apply symbol transformation rules to input sequences",        rules=rules,        examples=examples,        test_items=test_items,        difficulty=difficulty,        n_rules=len(rules),        domain="symbol",    )def generate_number_system(seed: str = "num_default", difficulty: int = 1) -> RuleSystem:    """    Generate a novel number system / arithmetic.    Input: expression in the invented system    Rules: how operators work    Output: numeric result    """    rng = _make_rng(seed)    op_names = ["grok", "flim", "zorp", "quex", "blix"]    ops = rng.sample(op_names, 3)    if difficulty == 1:        # Two operators: basic arithmetic with twist        a_op, b_op = ops[0], ops[1]        a_fn = lambda x, y: x + y + 1  # "grok" = add and increment        b_fn = lambda x, y: abs(x - y)  # "flim" = absolute difference        rules = [            f"'{a_op}(x, y)' means: add x and y, then add 1",            f"'{b_op}(x, y)' means: absolute difference of x and y",        ]        op_map = {a_op: a_fn, b_op: b_fn}    elif difficulty == 2:        a_op, b_op, c_op = ops[0], ops[1], ops[2]        a_fn = lambda x, y: x * 2 + y        b_fn = lambda x, y: (x + y) % 10        c_fn = lambda x, y: max(x, y) - min(x, y) + 1        rules = [            f"'{a_op}(x, y)' means: double x, then add y",            f"'{b_op}(x, y)' means: add x and y, take the last digit (mod 10)",            f"'{c_op}(x, y)' means: difference of larger and smaller, plus 1",        ]        op_map = {a_op: a_fn, b_op: b_fn, c_op: c_fn}    else:  # difficulty == 3        a_op, b_op, c_op = ops[0], ops[1], ops[2]        # Nested operations        a_fn = lambda x, y: x + y + 1        b_fn = lambda x, y: x * y        rules = [            f"'{a_op}(x, y)' means: add x and y, then add 1",            f"'{b_op}(x, y)' means: multiply x and y",            f"Operations can be nested: '{a_op}({b_op}(x, y), z)' means: first compute {b_op}(x, y), then use the result as the first argument to {a_op}",        ]        op_map = {a_op: a_fn, b_op: b_fn}    # Generate examples    all_items = []    for _ in range(20):        if difficulty <= 2:            op_name = rng.choice(list(op_map.keys()))            x = rng.randint(1, 9)            y = rng.randint(1, 9)            result = op_map[op_name](x, y)            expr = f"{op_name}({x}, {y})"        else:            # Allow nesting            if rng.random() < 0.5:                op_name = rng.choice(list(op_map.keys()))                x = rng.randint(1, 9)                y = rng.randint(1, 9)                result = op_map[op_name](x, y)                expr = f"{op_name}({x}, {y})"            else:                inner_op = rng.choice(list(op_map.keys()))                outer_op = rng.choice(list(op_map.keys()))                x, y, z = rng.randint(1, 5), rng.randint(1, 5), rng.randint(1, 5)                inner_result = op_map[inner_op](x, y)                result = op_map[outer_op](inner_result, z)                expr = f"{outer_op}({inner_op}({x}, {y}), {z})"        all_items.append({"input": expr, "output": str(result)})    # Deduplicate    seen = set()    unique_items = []    for item in all_items:        if item["input"] not in seen:            seen.add(item["input"])            unique_items.append(item)    rng.shuffle(unique_items)    n_ex = min(12, len(unique_items) - 5)    examples = unique_items[:n_ex]    test_items = unique_items[n_ex:n_ex + 5]    return RuleSystem(        name=f"NumberSystem-{seed}",        description="Evaluate expressions using novel arithmetic operators",        rules=rules,        examples=examples,        test_items=test_items,        difficulty=difficulty,        n_rules=len(rules),        domain="number",    )# Pre-generated systems for the benchmarkLEARNING_CURVE_SYSTEMS = [    generate_symbol_system("lc_sym_easy", difficulty=1),    generate_symbol_system("lc_sym_med", difficulty=2),    generate_symbol_system("lc_sym_hard", difficulty=3),    generate_number_system("lc_num_easy", difficulty=1),    generate_number_system("lc_num_med", difficulty=2),    generate_number_system("lc_num_hard", difficulty=3),]# Systems for transfer testingTRANSFER_BASE_SYSTEM = generate_symbol_system("transfer_base", difficulty=2)TRANSFER_NEAR_SYSTEM = generate_symbol_system("transfer_near", difficulty=2)TRANSFER_FAR_SYSTEM = generate_number_system("transfer_far", difficulty=2)# Systems for interference testingINTERFERENCE_A = generate_symbol_system("interf_a", difficulty=2)INTERFERENCE_B = generate_symbol_system("interf_b_similar", difficulty=2)

In [ ]:
"""Learning Benchmark 1: Novel Rule System Learning CurvesMeasures how model performance improves with increasing numbers oftraining examples — the fundamental learning curve.Protocol:1. Present a novel rule system description (rules only, no examples)2. Incrementally provide training examples: 0, 2, 4, 8, 12 examples3. At each step, test on held-out problems4. Plot accuracy vs. number of training examples5. Measure learning curve shape and efficiencyCognitive Science Basis:- Power Law of Practice (Newell & Rosenbloom, 1981)- Learning curves (Bryan & Harter, 1897)- Sample efficiency as a measure of learning abilityKey Innovation:- Rule systems are procedurally generated → not in training data- Tests genuine in-context learning, not memorization- Multiple difficulty levels test learning capacityScore: Composite of learning rate, asymptotic accuracy, and sample efficiency."""import kaggle_benchmarks as kbenchfrom dataclasses import dataclassimport numpy as npimport reimport json# LEARNING_CURVE_SYSTEMS defined above@dataclassclass RuleAnswer:    answer: str    reasoning: str# Test checkpoints: how many examples to show before each testCHECKPOINTS = [0, 2, 4, 8, 12]def normalize_output(text: str) -> str:    """Normalize output for comparison."""    text = text.strip().lower()    text = re.sub(r'\s+', ' ', text)    return textdef check_output(model_output: str, expected: str) -> bool:    """Check if model output matches expected."""    m = normalize_output(model_output)    e = normalize_output(expected)    # Exact match or containment    return e in m or m in edef fit_power_law(x: np.ndarray, y: np.ndarray) -> tuple[float, float, float]:    """    Fit y = a * x^b + c (power law of practice).    Returns (a, b, c) or best-effort approximation.    """    # Simple: use log-linear regression on non-zero x    mask = x > 0    if mask.sum() < 2:        return (0.0, 0.0, float(y[0]) if len(y) > 0 else 0.0)    log_x = np.log(x[mask])    # Subtract baseline (zero-shot)    baseline = y[0] if len(y) > 0 else 0    y_adj = y[mask] - baseline    y_adj = np.maximum(y_adj, 0.001)  # Avoid log(0)    log_y = np.log(y_adj)    # Linear regression in log space    try:        b, log_a = np.polyfit(log_x, log_y, 1)        a = np.exp(log_a)        return (float(a), float(b), float(baseline))    except Exception:        return (0.0, 0.0, float(baseline))@kbench.task(name="learning_curves")def learning_curves(llm) -> float:    """    Learning Curves Benchmark.    Tests how model performance improves with training examples    for novel rule systems that cannot be in training data.    Score = weighted average of:      0.30 * mean_asymptotic_accuracy      0.30 * mean_learning_rate (normalized)      0.20 * mean_sample_efficiency (normalized)      0.20 * curve_quality (does it show genuine learning?)    """    all_curves = []    results_log = []    for system in LEARNING_CURVE_SYSTEMS:        curve = {"system": system.name, "difficulty": system.difficulty, "checkpoints": []}        for n_examples in CHECKPOINTS:            # Only use up to n_examples from the pool            n_examples_actual = min(n_examples, len(system.examples))            with kbench.chats.new(f"{system.name}_n{n_examples}"):                # Build prompt with rules + n examples                prompt_parts = [                    f"You are learning the rule system: **{system.name}**\n",                    f"Description: {system.description}\n",                    "\n**Rules:**",                ]                for rule in system.rules:                    prompt_parts.append(f"- {rule}")                if n_examples_actual > 0:                    prompt_parts.append(f"\n**Training examples ({n_examples_actual}):**")                    for ex in system.examples[:n_examples_actual]:                        prompt_parts.append(f"  Input: {ex['input']}  →  Output: {ex['output']}")                # Test on held-out items                n_correct = 0                for ti, test_item in enumerate(system.test_items):                    test_prompt = "\n".join(prompt_parts) + (                        f"\n\nNow apply the rules to this new input:\n"                        f"Input: {test_item['input']}\n\n"                        f"Respond with ONLY a JSON object:\n"                        f'{{"answer": "<output after applying rules>", "reasoning": "<your steps>"}}'                    )                    try:                        result = llm.prompt(test_prompt, schema=RuleAnswer)                        answer = result.answer                    except Exception:                        raw = llm.prompt(test_prompt)                        try:                            parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())                            answer = str(parsed.get("answer", raw))                        except Exception:                            answer = raw                    if check_output(answer, test_item["output"]):                        n_correct += 1                accuracy = n_correct / len(system.test_items) if system.test_items else 0                curve["checkpoints"].append({                    "n_examples": n_examples,                    "accuracy": accuracy,                    "n_correct": n_correct,                    "n_total": len(system.test_items),                })        all_curves.append(curve)    # ── Compute Metrics ──    asymptotic_accs = []    learning_rates = []    sample_efficiencies = []    curve_qualities = []    for curve in all_curves:        checkpoints = curve["checkpoints"]        x = np.array([c["n_examples"] for c in checkpoints], dtype=float)        y = np.array([c["accuracy"] for c in checkpoints], dtype=float)        # Asymptotic accuracy (last checkpoint)        asymptotic = float(y[-1])        asymptotic_accs.append(asymptotic)        # Learning rate: improvement from 0 to max examples        learning_rate = float(y[-1] - y[0]) if len(y) > 1 else 0        learning_rates.append(max(0, learning_rate))        # Sample efficiency: first checkpoint where accuracy >= 0.8 (lower = better)        efficiency = len(CHECKPOINTS)  # Default: never reached        for i, c in enumerate(checkpoints):            if c["accuracy"] >= 0.8:                efficiency = i                break        # Normalize: 0 = worst (never), 1 = best (zero-shot)        sample_efficiencies.append(1 - efficiency / len(CHECKPOINTS))        # Curve quality: is there monotonic improvement? (genuine learning)        if len(y) > 1:            improvements = np.diff(y)            # Fraction of steps that show improvement or maintenance            quality = np.mean(improvements >= -0.05)  # Allow tiny dips        else:            quality = 0.5        curve_qualities.append(float(quality))    # Overall score    mean_asymptotic = np.mean(asymptotic_accs)    mean_lr = np.mean(learning_rates)    mean_se = np.mean(sample_efficiencies)    mean_cq = np.mean(curve_qualities)    score = round(        0.30 * mean_asymptotic + 0.30 * mean_lr + 0.20 * mean_se + 0.20 * mean_cq,        4    )    # ── Logging ──    print(f"\n{'='*60}")    print(f"LEARNING CURVES BENCHMARK RESULTS")    print(f"{'='*60}")    print(f"Systems tested: {len(LEARNING_CURVE_SYSTEMS)}")    print(f"Checkpoints: {CHECKPOINTS}")    for curve in all_curves:        print(f"\n--- {curve['system']} (difficulty={curve['difficulty']}) ---")        for cp in curve["checkpoints"]:            bar = "█" * int(cp["accuracy"] * 20)            print(f"  n={cp['n_examples']:2d}: {cp['accuracy']:.2%} ({cp['n_correct']}/{cp['n_total']}) {bar}")    print(f"\n--- Aggregate Metrics ---")    print(f"Mean asymptotic accuracy: {mean_asymptotic:.3f}")    print(f"Mean learning rate:       {mean_lr:.3f}")    print(f"Mean sample efficiency:   {mean_se:.3f}")    print(f"Mean curve quality:       {mean_cq:.3f}")    print(f"Composite score:          {score:.4f}")    # Per-system summary    print(f"\n--- Per-System Summary ---")    for i, curve in enumerate(all_curves):        print(f"  {curve['system']}: asym={asymptotic_accs[i]:.2f}, "              f"lr={learning_rates[i]:.2f}, se={sample_efficiencies[i]:.2f}, "              f"cq={curve_qualities[i]:.2f}")    return score# ─── Run ────────────────────────────────────────────────────────────learning_curves.run(llm=kbench.llm)